# Construction de l'indice SCO-RISK

Ce notebook transforme le fichier nettoyé en un indice de risque transparent de 0 à 100. L'objectif n'est pas de déclarer qu'un enfant abandonnera, mais de classer les provinces qui nécessitent une revue prioritaire.

## Méthode

Pour chaque année, les provinces sont classées sur trois dimensions : abandon, redoublement et non-achèvement régional. Le scénario principal utilise les poids 50 %, 30 % et 20 %. Deux scénarios alternatifs testent si le classement change fortement lorsque les poids changent.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

RACINE = Path.cwd()
if RACINE.name == '03_transformation':
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / 'scripts'))
from transformer_indicateurs import transformer, POIDS

rapport = transformer(
    RACINE / '02_nettoyage/SCO-RISKBurkina_02-nettoyage_D1_engineer_donnees-decrochage-province-annee_v1.csv',
    RACINE / '03_transformation/SCO-RISKBurkina_03-transformation_D1_engineer_indicateurs-province-annee_v1.csv',
    RACINE / '03_transformation/SCO-RISKBurkina_03-transformation_D1_data-scientist_sensibilite-ponderations_v1.csv',
    RACINE / 'rapports_prives/06_qualite_indice_risque.json',
)
rapport

In [ ]:
indicateurs = pd.read_csv(RACINE / '03_transformation/SCO-RISKBurkina_03-transformation_D1_engineer_indicateurs-province-annee_v1.csv')
sensibilite = pd.read_csv(RACINE / '03_transformation/SCO-RISKBurkina_03-transformation_D1_data-scientist_sensibilite-ponderations_v1.csv')
assert not indicateurs.duplicated(['province', 'annee_scolaire']).any()
assert indicateurs['score_risque'].between(0, 100).all()
print('Dimensions :', indicateurs.shape)
print('Poids testés :', POIDS)
sensibilite

In [ ]:
# Revue locale du classement le plus récent.
colonnes = ['province', 'region_2025', 'score_risque', 'niveau_alerte', 'fiabilite']
indicateurs[indicateurs['annee_scolaire'] == '2024-2025'].nlargest(15, 'score_risque')[colonnes]

## Limites

L'abandon et le redoublement sont actuellement des moyennes non pondérées des taux par niveau. L'achèvement est régional. Le score est donc un outil de priorisation explicable, pas une probabilité individuelle.